In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import numpy as np

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# 1. 合并数据集
train_df = pd.read_csv('./data/Model_construction/train.csv')
validation_df = pd.read_csv('./data/Model_construction/validation.csv')

In [4]:
combined_df = pd.concat([train_df, validation_df], axis=0).reset_index(drop=True)
combined_df['CTgene'] = combined_df['CTgene'].str.replace('_', '.', regex=False)

In [7]:
df = combined_df

In [8]:
combined_df = df

In [8]:
# Load metadata and merge patient_id
df1 = pd.read_csv('./data/metadata/input_meta.csv')[['ID', 'Label', 'patient_id']]
df1 = df1[df1['ID'].isin(combined_df['ID'])]
combined_df = combined_df.merge(df1[['ID', 'patient_id']], on='ID', how='left')
# 计算每个患者的样本数和正例比例
patient_stats = df1.groupby('patient_id').agg(
    total_samples=('Label', 'count'),
    positive_ratio=('Label', lambda x: (x == 'Neo').mean())
).reset_index()

In [9]:
# 添加每个患者的样本ID列表（用于后续分割）
patient_ids_to_samples = df1.groupby('patient_id')['ID'].apply(list).to_dict()
patient_stats['samples'] = patient_stats['patient_id'].map(patient_ids_to_samples)
test_size=0.2
max_iter=10000
# 设置随机种子以确保可重复性
np.random.seed(666)

In [10]:
best_diff = float('inf')
best_train = None
best_test = None

In [11]:
# 总体正例比例参考值
overall_positive_ratio = (df1['Label'] == 'Neo').mean()
for _ in range(max_iter):
    # 随机打乱患者
    shuffled = patient_stats.sample(frac=1)  
    # 按打乱顺序累积患者
    cum_samples = 0
    cum_positive = 0
    train_patients = []   
    for _, row in shuffled.iterrows():
        # 计算当前患者加入后的统计量
        new_total = cum_samples + row['total_samples']
        new_positive = cum_positive + row['positive_ratio'] * row['total_samples']     
        # 检查是否超过测试集大小容差
        if abs(1 - new_total / len(df1) - test_size) > 0.05:  # 5%容差
            train_patients.append(row)
            cum_samples = new_total
            cum_positive = new_positive
        else:
            break 
    # 创建测试集（剩余患者）
    test_patients = shuffled[~shuffled['patient_id'].isin([p['patient_id'] for p in train_patients])]  
    # 计算训练集正例比例
    if cum_samples == 0:
        train_ratio = 0
    else:
        train_ratio = cum_positive / cum_samples  
    # 计算测试集正例比例
    test_positive = (df1[df1['patient_id'].isin(test_patients['patient_id'])]['Label'] == 'Neo').sum()
    test_ratio = test_positive / (len(df1) - cum_samples) if cum_samples < len(df1) else 0 
    # 评估分布差异
    diff = abs(train_ratio - test_ratio)  
    # 检查是否找到更好的分割
    if diff < best_diff:
        best_diff = diff
        best_train = train_patients
        best_test = test_patients

In [12]:
# 提取样本ID
train_ids = [sample for p in best_train for sample in p['samples']]
try:
    test_ids = [sample for p in best_test for sample in p['samples']]
except:
    test_ids = []
    for sample in best_test['samples']:
        test_ids.extend((sample))

In [13]:
train_set = df[df['ID'].isin(train_ids)].copy()
test_set = df[df['ID'].isin(test_ids)].copy()
# 验证结果
print(f"训练集大小: {len(train_set)} 样本")
print(f"测试集大小: {len(test_set)} 样本")
print(f"训练集正例比例: {100*(train_set['Label']=='Neo').mean():.2f}%")
print(f"测试集正例比例: {100*(test_set['Label']=='Neo').mean():.2f}%")
# print(f"训练集患者数: {train_set['patient_id'].nunique()}")
# print(f"测试集患者数: {test_set['patient_id'].nunique()}")
train_index = train_set.index
test_index = test_set.index

训练集大小: 31211 样本
测试集大小: 14455 样本
训练集正例比例: 26.88%
测试集正例比例: 26.81%


In [14]:
def process_CTgene(df):
    default_TRBD = "NA"
    split_genes = df['CTgene'].str.split('.', expand=True)
    split_genes = split_genes.reindex(columns=[0, 1, 2, 3, 4], fill_value=default_TRBD)
    split_genes.columns = ['TRAV', 'TRAJ', 'TRBV', 'TRBD', 'TRBJ']
    split_genes['TRBD'] = split_genes['TRBD'].apply(lambda x: x if isinstance(x, str) and x.startswith("TRBD") else default_TRBD)
    split_genes['TRBJ'] = split_genes['TRBJ'].apply(lambda x: x if isinstance(x, str) and not x.startswith("TRBD") else default_TRBD)
    df[['TRAV', 'TRAJ', 'TRBV', 'TRBD', 'TRBJ']] = split_genes
    return df

In [15]:
def create_gene_index_mapping(df, column_name, unknown_token='UNK'):
    gene_types = df[column_name].astype(str).unique()
    gene_to_idx = {gene: idx for idx, gene in enumerate(gene_types)}
    gene_to_idx[unknown_token] = len(gene_to_idx)
    return gene_to_idx

In [16]:
def encode_cdr3_sequence_properties(seq, max_len):
    non_polar_non_aromatic = ['A', 'I', 'L', 'M', 'V', 'G', 'P']
    aromatic = ['F', 'W', 'Y']
    polar_non_aromatic = ['S', 'T', 'N', 'Q', 'C']
    negative_charged = ['D', 'E']
    positive_charged = ['R', 'K', 'H']
    features = np.zeros(max_len)
    for i, aa in enumerate(seq):
        if i >= max_len:
            break
        if aa in non_polar_non_aromatic:
            features[i] = 1
        elif aa in aromatic:
            features[i] = 2
        elif aa in polar_non_aromatic:
            features[i] = 3
        elif aa in negative_charged:
            features[i] = 4
        elif aa in positive_charged:
            features[i] = 5
    return features

In [17]:
def pretreatment(df, trav_to_idx=None, traj_to_idx=None, trbv_to_idx=None, trbd_to_idx=None, trbj_to_idx=None, cell_to_idx=None, is_train=False):
    df['CTgene'] = df['CTgene'].str.replace('_', '.', regex=False)
    if 'patient_id' in df.columns:
        gene_columns = df.columns[df.columns.get_loc('pMT'): df.columns.get_loc('patient_id')]
    else:
        gene_columns = df.columns[df.columns.get_loc('pMT'):]
    X_genes = df[gene_columns].values.astype(float)
    if 'Label' in df.columns:
        df['Label_num'] = df['Label'].map({'Neo': 1, 'nonNeo': 0})
        y = df['Label_num'].values.copy()
        df = df.drop(labels=['Label_num', 'Label'], axis=1)
    else:
        y = None
    if df['HPV'].dtype == object:
        df['HPV_num'] = df['HPV'].map({'HPV+': 1, 'HPV-': 0})
    else:
        df['HPV_num'] = df['HPV']
    if cell_to_idx is None or is_train:
        cell_types = df['cell_names'].astype(str).unique()
        cell_to_idx = {cell: idx for idx, cell in enumerate(cell_types)}
    df['celltype_idx'] = df['cell_names'].astype(str).map(cell_to_idx).fillna(-1).astype(int)
    df = process_CTgene(df)
    if trav_to_idx is None or is_train:
        trav_to_idx = create_gene_index_mapping(df, 'TRAV')
    if traj_to_idx is None or is_train:
        traj_to_idx = create_gene_index_mapping(df, 'TRAJ')
    if trbv_to_idx is None or is_train:
        trbv_to_idx = create_gene_index_mapping(df, 'TRBV')
    if trbd_to_idx is None or is_train:
        trbd_to_idx = create_gene_index_mapping(df, 'TRBD')
    if trbj_to_idx is None or is_train:
        trbj_to_idx = create_gene_index_mapping(df, 'TRBJ')
    df['TRAV_idx'] = df['TRAV'].map(trav_to_idx).fillna(trav_to_idx['UNK']).astype(int)
    df['TRAJ_idx'] = df['TRAJ'].map(traj_to_idx).fillna(traj_to_idx['UNK']).astype(int)
    df['TRBV_idx'] = df['TRBV'].map(trbv_to_idx).fillna(trbv_to_idx['UNK']).astype(int)
    df['TRBD_idx'] = df['TRBD'].map(trbd_to_idx).fillna(trbd_to_idx['UNK']).astype(int)
    df['TRBJ_idx'] = df['TRBJ'].map(trbj_to_idx).fillna(trbj_to_idx['UNK']).astype(int)
    MAX_LEN_ALPHA = 21
    MAX_LEN_BETA = 26
    df['seq_alpha_encoded_props'] = df['cdr3_aa1'].apply(lambda s: encode_cdr3_sequence_properties(s, MAX_LEN_ALPHA))
    df['seq_beta_encoded_props'] = df['cdr3_aa2'].apply(lambda s: encode_cdr3_sequence_properties(s, MAX_LEN_BETA))
    df['seq_alpha_len'] = df['cdr3_aa1'].apply(len)
    df['seq_beta_len'] = df['cdr3_aa2'].apply(len)
    X_seq_alpha_props = df['seq_alpha_encoded_props'].apply(lambda x: torch.tensor(x, dtype=torch.long)).tolist()
    X_seq_beta_props = df['seq_beta_encoded_props'].apply(lambda x: torch.tensor(x, dtype=torch.long)).tolist()
    X_seq_alpha_len = df['seq_alpha_len'].values
    X_seq_beta_len = df['seq_beta_len'].values
    X_ctgene = df[['TRAV_idx', 'TRAJ_idx', 'TRBV_idx', 'TRBD_idx', 'TRBJ_idx']].values
    X_celltype = df['celltype_idx'].values
    X_hpvinf = df['HPV_num'].values
    to_idx = [trav_to_idx, traj_to_idx, trbv_to_idx, trbd_to_idx, trbj_to_idx, cell_to_idx]
    return to_idx, X_genes, X_ctgene, X_celltype, X_hpvinf, X_seq_alpha_props, X_seq_beta_props, X_seq_alpha_len, X_seq_beta_len, y

In [18]:
class TCellDataset(Dataset):
    def __init__(self, X_gene, X_ct, X_cell, X_hpv, X_seq_a_props, X_seq_b_props, X_seq_a_len, X_seq_b_len, y=None):
        self.X_gene = torch.tensor(X_gene, dtype=torch.float32)
        self.X_ctgene = torch.tensor(X_ct, dtype=torch.long)
        self.X_cell = torch.tensor(X_cell, dtype=torch.long)
        self.X_hpv = torch.tensor(X_hpv, dtype=torch.long)
        self.X_seq_alpha_props = torch.tensor(np.array(X_seq_a_props), dtype=torch.long)
        self.X_seq_beta_props = torch.tensor(np.array(X_seq_b_props), dtype=torch.long)
        self.X_seq_alpha_len = torch.tensor(np.array(X_seq_a_len), dtype=torch.float32)
        self.X_seq_beta_len = torch.tensor(np.array(X_seq_b_len), dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long) if y is not None else None

    def __len__(self):
        return len(self.X_gene)

    def __getitem__(self, idx):
        item = (self.X_gene[idx], self.X_ctgene[idx], self.X_cell[idx], self.X_hpv[idx],
                self.X_seq_alpha_props[idx], self.X_seq_beta_props[idx], self.X_seq_alpha_len[idx], self.X_seq_beta_len[idx])
        if self.y is not None:
            return item + (self.y[idx],)
        return item

In [19]:
train_set = combined_df[combined_df['ID'].isin(train_ids)].copy()
test_set = combined_df[combined_df['ID'].isin(test_ids)].copy()

In [20]:
# Preprocess with is_train=True for training data to create mappings
to_idx_train, Xg_train, ct_train, cell_train, hpv_train, seqA_props_train, seqB_props_train, seqA_len_train, seqB_len_train, y_train = pretreatment(train_set, is_train=True)
trav_to_idx, traj_to_idx, trbv_to_idx, trbd_to_idx, trbj_to_idx, cell_to_idx = to_idx_train

In [21]:
# For test, use same mappings
to_idx_test, Xg_test, ct_test, cell_test, hpv_test, seqA_props_test, seqB_props_test, seqA_len_test, seqB_len_test, y_test = pretreatment(test_set, 
    trav_to_idx=trav_to_idx, traj_to_idx=traj_to_idx, trbv_to_idx=trbv_to_idx, trbd_to_idx=trbd_to_idx, trbj_to_idx=trbj_to_idx, cell_to_idx=cell_to_idx)

In [22]:
# 数据集和 loader
train_dataset = TCellDataset(Xg_train, ct_train, cell_train, hpv_train, seqA_props_train, seqB_props_train, seqA_len_train, seqB_len_train, y_train)
test_dataset = TCellDataset(Xg_test, ct_test, cell_test, hpv_test, seqA_props_test, seqB_props_test, seqA_len_test, seqB_len_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [23]:
# 模型参数
n_genes = Xg_train.shape[1]
n_trav = len(trav_to_idx)
n_traj = len(traj_to_idx)
n_trbv = len(trbv_to_idx)
n_trbd = len(trbd_to_idx)
n_trbj = len(trbj_to_idx)
n_celltype = len(cell_to_idx)
n_hpvinf = 2

In [24]:
class TCellClassifier(nn.Module):
    def __init__(self, n_genes, n_trav, n_traj, n_trbv, n_trbd, n_trbj, n_celltype, n_hpvinf,
                 property_vocab_size=6, max_len_a=21, max_len_b=26,
                 gene_hidden=128, embed_dim_aa_props=32, embed_dim_ct=32, embed_dim_cell=8, embed_dim_hp=2,
                 num_heads=4, num_transformer_layers=2):
        super(TCellClassifier, self).__init__()
        self.embed_trav = nn.Embedding(n_trav, embed_dim_ct)
        self.embed_traj = nn.Embedding(n_traj, embed_dim_ct)
        self.embed_trbv = nn.Embedding(n_trbv, embed_dim_ct)
        self.embed_trbd = nn.Embedding(n_trbd, embed_dim_ct)
        self.embed_trbj = nn.Embedding(n_trbj, embed_dim_ct)
        self.embed_cell = nn.Embedding(n_celltype, embed_dim_cell)
        self.embed_hpv = nn.Embedding(n_hpvinf, embed_dim_hp)
        self.embed_aa_properties = nn.Embedding(property_vocab_size, embed_dim_aa_props, padding_idx=0)
        
        # Transformer for CDR3
        self.transformer_alpha = nn.TransformerEncoderLayer(d_model=embed_dim_aa_props, nhead=num_heads, batch_first=True)
        self.transformer_beta = nn.TransformerEncoderLayer(d_model=embed_dim_aa_props, nhead=num_heads, batch_first=True)
        self.transformer_encoder_alpha = nn.TransformerEncoder(self.transformer_alpha, num_layers=num_transformer_layers)
        self.transformer_encoder_beta = nn.TransformerEncoder(self.transformer_beta, num_layers=num_transformer_layers)
        
        # Gene FC
        self.fc_genes = nn.Sequential(
            nn.Linear(n_genes, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, gene_hidden),
        )
        
        # 修正 combined_size
        combined_size = gene_hidden + embed_dim_aa_props + embed_dim_aa_props + 5 * embed_dim_ct + embed_dim_cell + embed_dim_hp + 2
        
        # MultiheadAttention with corrected embed_dim
        self.multihead_attn = nn.MultiheadAttention(embed_dim=combined_size, num_heads=num_heads, batch_first=True)
        
        self.fc_comb1 = nn.Linear(combined_size, 128)
        self.fc_comb2 = nn.Linear(128, 64)
        self.fc_out = nn.Linear(64, 2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, gene_input, ct_input, cell_input, hpv_input, seq_alpha_props, seq_beta_props, seq_alpha_len, seq_beta_len):
        x_gene = self.fc_genes(gene_input)
        x_gene = self.relu(x_gene)
        
        x_trav = self.embed_trav(ct_input[:, 0])
        x_traj = self.embed_traj(ct_input[:, 1])
        x_trbv = self.embed_trbv(ct_input[:, 2])
        x_trbd = self.embed_trbd(ct_input[:, 3])
        x_trbj = self.embed_trbj(ct_input[:, 4])
        x_ct = torch.cat([x_trav, x_traj, x_trbv, x_trbd, x_trbj], dim=1)
        
        x_cell = self.embed_cell(cell_input)
        x_hpv = self.embed_hpv(hpv_input)
        
        x_alpha_seq = self.embed_aa_properties(seq_alpha_props)
        h_alpha = self.transformer_encoder_alpha(x_alpha_seq).mean(dim=1)
        
        x_beta_seq = self.embed_aa_properties(seq_beta_props)
        h_beta = self.transformer_encoder_beta(x_beta_seq).mean(dim=1)
        
        seq_alpha_len = seq_alpha_len.unsqueeze(1)
        seq_beta_len = seq_beta_len.unsqueeze(1)
        
        x_combined = torch.cat([x_gene, h_alpha, h_beta, x_ct, x_cell, x_hpv, seq_alpha_len, seq_beta_len], dim=1)
        print("x_combined shape:", x_combined.shape)  # 调试打印
        
        x_attn, _ = self.multihead_attn(x_combined.unsqueeze(1), x_combined.unsqueeze(1), x_combined.unsqueeze(1))
        x_attn = x_attn.squeeze(1)
        
        x = self.fc_comb1(x_attn)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc_comb2(x)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc_out(x)
        return logits

In [25]:
model = TCellClassifier(n_genes, n_trav, n_traj, n_trbv, n_trbd, n_trbj, n_celltype, n_hpvinf).to(device)

In [26]:
# 损失和优化器
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

In [27]:
# 训练循环
train_losses = []
test_losses = []
best_test_loss = float('inf')
epochs = 50

In [28]:
from tqdm import tqdm 
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
import os
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import pickle
import shap
os.makedirs('results', exist_ok=True)
os.makedirs('results/confusion_matrices', exist_ok=True)

C:\Users\wenzh\anaconda3\envs\deep_learning_TR\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    all_train_preds = []
    all_train_labels = []
    if len(train_loader) == 0:
        print(f"Epoch {epoch}: 警告：train_loader 为空。跳过本轮训练。")
    else:
        train_loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch} 训练")
        for batch_idx, batch in enumerate(train_loop):
            try:
                Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl, labels = batch
            except ValueError as e:
                print(f"Epoch {epoch}, Batch {batch_idx}: 解包 batch 错误：{e}")
                continue
            Xg = Xg.to(device)
            Xct = Xct.to(device)
            Xcell = Xcell.to(device)
            Xhpv = Xhpv.to(device)
            Xa = Xa.to(device)
            Xb = Xb.to(device)
            Xal = Xal.to(device)
            Xbl = Xbl.to(device)
            labels = labels.to(device)
            logits = model(Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * Xg.size(0)
            preds = torch.argmax(logits, dim=1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())
            train_loop.set_postfix(loss=loss.item())
        train_loss = running_loss / len(train_dataset) if len(train_dataset) > 0 else 0
        train_losses.append(train_loss)
    model.eval()
    test_loss = 0.0
    all_test_preds = []
    all_test_labels = []
    if len(test_loader) == 0:
        print(f"Epoch {epoch}: 警告：test_loader 为空。跳过本轮验证。")
    else:
        with torch.no_grad():
            test_loop = tqdm(test_loader, leave=False, desc=f"Epoch {epoch} 测试")
            for batch_idx, batch in enumerate(test_loop):
                Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl, labels = batch
                Xg = Xg.to(device); Xct = Xct.to(device); Xcell = Xcell.to(device)
                Xhpv = Xhpv.to(device); Xa = Xa.to(device); Xb = Xb.to(device)
                Xal = Xal.to(device); Xbl = Xbl.to(device)
                labels = labels.to(device)
                logits = model(Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl)
                loss = criterion(logits, labels)
                test_loss += loss.item() * Xg.size(0)
                preds = torch.argmax(logits, dim=1)
                all_test_preds.extend(preds.cpu().numpy())
                all_test_labels.extend(labels.cpu().numpy())
                current_acc = (preds == labels).sum().item() / labels.size(0)
                test_loop.set_postfix(loss=loss.item(), acc=current_acc)
        avg_test_loss = test_loss / len(test_dataset) if len(test_dataset) > 0 else 0
        test_losses.append(avg_test_loss)
        test_acc = np.mean(np.array(all_test_preds) == np.array(all_test_labels)) if all_test_labels else 0
        # 保存最佳模型
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"Saved best model at epoch {epoch} with test loss {avg_test_loss:.4f}")
print("训练完成。")

Epoch 1 训练:   0%|                                                                            | 0/244 [00:00<?, ?it/s]

x_combined shape: torch.Size([128, 364])


Epoch 1 训练:   1%|▋                                                       | 3/244 [00:00<00:58,  4.12it/s, loss=0.693]

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])


Epoch 1 训练:   2%|█▏                                                      | 5/244 [00:01<00:38,  6.26it/s, loss=0.688]

...

Epoch 50 测试:  91%|████████████████████████████████████████    | 103/113 [00:03<00:00, 29.45it/s, acc=1, loss=0.00196]

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])


Epoch 50 测试:  96%|█████████████████████████████████████████▍ | 109/113 [00:03<00:00, 28.80it/s, acc=0.484, loss=2.57]

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])


x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([119, 364])
训练完成。


In [49]:
# 加载交叉验证分组
cv_df = pd.read_csv('交叉验证分组.csv')
folds = 5  # 5 折

In [ ]:
# 交叉验证循环
all_roc_aucs = []
all_pr_aucs = []
all_test_accuracies = []
all_test_f1s = []
for fold in range(1, folds + 1):
    print(f"\n=== Fold {fold} ===")
    train_col = f'Train_{fold}'
    test_col = f'Test_{fold}'
    
    train_patients = cv_df[train_col].dropna().tolist()
    test_patients = cv_df[test_col].dropna().tolist()
    
    train_ids = [sid for pid in train_patients for sid in patient_ids_to_samples.get(pid, [])]
    test_ids = [sid for pid in test_patients for sid in patient_ids_to_samples.get(pid, [])]
    
    train_set = combined_df[combined_df['ID'].isin(train_ids)].copy()
    test_set = combined_df[combined_df['ID'].isin(test_ids)].copy()
    
    # 预处理
    to_idx_train, Xg_train, ct_train, cell_train, hpv_train, seqA_props_train, seqB_props_train, seqA_len_train, seqB_len_train, y_train = pretreatment(train_set, is_train=True)
    trav_to_idx, traj_to_idx, trbv_to_idx, trbd_to_idx, trbj_to_idx, cell_to_idx = to_idx_train
    
    to_idx_test, Xg_test, ct_test, cell_test, hpv_test, seqA_props_test, seqB_props_test, seqA_len_test, seqB_len_test, y_test = pretreatment(test_set, 
        trav_to_idx=trav_to_idx, traj_to_idx=traj_to_idx, trbv_to_idx=trbv_to_idx, trbd_to_idx=trbd_to_idx, trbj_to_idx=trbj_to_idx, cell_to_idx=cell_to_idx)
    
    # 保存映射（用于 SHAP）
    with open(f'fold_{fold}_mappings.pkl', 'wb') as f:
        pickle.dump(to_idx_train, f)
    
    # 数据集
    train_dataset = TCellDataset(Xg_train, ct_train, cell_train, hpv_train, seqA_props_train, seqB_props_train, seqA_len_train, seqB_len_train, y_train)
    test_dataset = TCellDataset(Xg_test, ct_test, cell_test, hpv_test, seqA_props_test, seqB_props_test, seqA_len_test, seqB_len_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
    
    # 模型参数
    n_genes = Xg_train.shape[1]
    n_trav = len(trav_to_idx)
    n_traj = len(traj_to_idx)
    n_trbv = len(trbv_to_idx)
    n_trbd = len(trbd_to_idx)
    n_trbj = len(trbj_to_idx)
    n_celltype = len(cell_to_idx)
    n_hpvinf = 2
    
    # 初始化模型
    model = TCellClassifier(n_genes, n_trav, n_traj, n_trbv, n_trbd, n_trbj, n_celltype, n_hpvinf).to(device)
    
    criterion = nn.CrossEntropyLoss().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    
    # 训练循环
    train_losses_fold = []
    test_losses_fold = []
    test_accuracies_fold = []
    test_f1s_fold = []
    best_test_loss = float('inf')
    epochs = 50
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        all_train_preds = []
        all_train_labels = []
        train_loop = tqdm(train_loader, leave=False, desc=f"Fold {fold} Epoch {epoch} 训练")
        for batch_idx, batch in enumerate(train_loop):
            Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl, labels = batch
            Xg = Xg.to(device)
            Xct = Xct.to(device)
            Xcell = Xcell.to(device)
            Xhpv = Xhpv.to(device)
            Xa = Xa.to(device)
            Xb = Xb.to(device)
            Xal = Xal.to(device)
            Xbl = Xbl.to(device)
            labels = labels.to(device)
            logits = model(Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * Xg.size(0)
            preds = torch.argmax(logits, dim=1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())
            train_loop.set_postfix(loss=loss.item())
        train_loss = running_loss / len(train_dataset)
        train_losses_fold.append(train_loss)
        
        model.eval()
        test_loss = 0.0
        all_test_preds = []
        all_test_labels = []
        all_test_probs = []  # 用于 ROC/PR
        with torch.no_grad():
            test_loop = tqdm(test_loader, leave=False, desc=f"Fold {fold} Epoch {epoch} 测试")
            for batch in test_loop:
                Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl, labels = batch
                Xg = Xg.to(device); Xct = Xct.to(device); Xcell = Xcell.to(device)
                Xhpv = Xhpv.to(device); Xa = Xa.to(device); Xb = Xb.to(device)
                Xal = Xal.to(device); Xbl = Xbl.to(device)
                labels = labels.to(device)
                logits = model(Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl)
                loss = criterion(logits, labels)
                test_loss += loss.item() * Xg.size(0)
                preds = torch.argmax(logits, dim=1)
                probs = torch.softmax(logits, dim=1)[:, 1]  # Neo 类概率
                all_test_preds.extend(preds.cpu().numpy())
                all_test_labels.extend(labels.cpu().numpy())
                all_test_probs.extend(probs.cpu().numpy())
        avg_test_loss = test_loss / len(test_dataset)
        
        # 评估性能
        test_acc = np.mean(np.array(all_test_preds) == np.array(all_test_labels))
        test_report = classification_report(all_test_labels, all_test_preds, output_dict=True)
        test_f1 = test_report['macro avg']['f1-score']
        test_f1s_fold.append(test_f1)
        test_accuracies_fold.append(test_acc)
        
        # ROC 曲线
        fpr, tpr, _ = roc_curve(all_test_labels, all_test_probs)
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'Fold {fold} Epoch {epoch} ROC Curve')
        plt.legend()
        plt.savefig(f'fold_{fold}_roc_epoch_{epoch}.png')
        plt.close()
        
        # PR 曲线
        precision, recall, _ = precision_recall_curve(all_test_labels, all_test_probs)
        pr_auc = auc(recall, precision)
        plt.figure()
        plt.plot(recall, precision, label=f'PR curve (AUPRC = {pr_auc:.2f})')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title(f'Fold {fold} Epoch {epoch} PR Curve')
        plt.legend()
        plt.savefig(f'fold_{fold}_pr_epoch_{epoch}.png')
        plt.close()
        
        # 混淆矩阵
        cm = confusion_matrix(all_test_labels, all_test_preds)
        plt.figure()
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Fold {fold} Epoch {epoch} Confusion Matrix')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.savefig(f'fold_{fold}_cm_epoch_{epoch}.png')
        plt.close()
        
        # Metrics evolution (每 epoch 绘制一次, 基于当前 fold)
        plt.figure()
        plt.plot(range(1, epoch+1), train_losses_fold[:epoch], label='Train Loss')
        plt.plot(range(1, epoch+1), [avg_test_loss] * epoch, label='Test Loss')  # 简化
        plt.plot(range(1, epoch+1), test_accuracies_fold[:epoch], label='Test Acc')
        plt.plot(range(1, epoch+1), test_f1s_fold[:epoch], label='Test F1')
        plt.xlabel('Epoch')
        plt.ylabel('Metric')
        plt.title(f'Fold {fold} Metrics Evolution')
        plt.legend()
        plt.savefig(f'fold_{fold}_metrics_evolution.png')
        plt.close()
        
        # 保存最佳模型
        if avg_test_loss < best_test_loss:
            best_test_loss = avg_test_loss
            torch.save(model.state_dict(), f'best_model_fold_{fold}.pth')

    # 记录折指标
    all_roc_aucs.append(roc_auc)
    all_pr_aucs.append(pr_auc)
    all_test_accuracies.append(test_acc)
    all_test_f1s.append(test_f1)


=== Fold 1 ===


Fold 1 Epoch 1 训练:   0%|▏                                                | 1/233 [00:00<01:11,  3.27it/s, loss=0.684]

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])


Fold 1 Epoch 1 训练:   2%|█                                                | 5/233 [00:00<00:22,  9.95it/s, loss=0.679]

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])


...

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])


Fold 5 Epoch 46 训练:  88%|██████████████████████████████████████▌     | 154/176 [00:11<00:01, 15.45it/s, loss=0.00045]

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])


Fold 5 Epoch 46 训练:  90%|██████████████████████████████████████▌    | 158/176 [00:12<00:01, 15.62it/s, loss=0.000176]

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])


In [58]:
# 交叉验证结束，汇总每个模型性能
print("\n=== 每个模型性能参数 ===")
for fold in range(1, folds + 1):
    print(f"Fold {fold}:")
    print(f"ROC AUC: {all_roc_aucs[fold-1]:.4f}")
    print(f"PR AUPRC: {all_pr_aucs[fold-1]:.4f}")
    print(f"测试准确率: {all_test_accuracies[fold-1]:.4f}")
    print(f"测试 F1: {all_test_f1s[fold-1]:.4f}")
    print("---")


=== 每个模型性能参数 ===
Fold 1:
ROC AUC: 0.9942
PR AUPRC: 0.9838
测试准确率: 0.9865
测试 F1: 0.9725
---
Fold 2:
ROC AUC: 0.9911
PR AUPRC: 0.9803
测试准确率: 0.9439
测试 F1: 0.9373
---
Fold 3:
ROC AUC: 0.9783
PR AUPRC: 0.9187
测试准确率: 0.9301
测试 F1: 0.8869
---
Fold 4:
ROC AUC: 0.9563
PR AUPRC: 0.9183
测试准确率: 0.7556
测试 F1: 0.6892
---
Fold 5:
ROC AUC: 0.9817
PR AUPRC: 0.9509
测试准确率: 0.9213
测试 F1: 0.9101
---


In [52]:
best_fold = np.argmax(all_test_f1s) + 1
print(f"最优折: Fold {best_fold}, F1: {all_test_f1s[best_fold-1]:.4f}")

最优折: Fold 1, F1: 0.9725


In [54]:
# 交叉验证结束，汇总平均指标
print("\n=== 交叉验证总结 ===")
print(f"平均 ROC AUC: {np.mean(all_roc_aucs):.4f}")
print(f"平均 PR AUPRC: {np.mean(all_pr_aucs):.4f}")
print(f"平均测试准确率: {np.mean(all_test_accuracies):.4f}")
print(f"平均测试 F1: {np.mean(all_test_f1s):.4f}")


=== 交叉验证总结 ===
平均 ROC AUC: 0.9803
平均 PR AUPRC: 0.9504
平均测试准确率: 0.9075
平均测试 F1: 0.8792


In [64]:
# 加载最优折的映射 (而非 params, 因为 params 从映射计算)
with open(f'fold_{best_fold}_mappings.pkl', 'rb') as f:
    to_idx_train = pickle.load(f)
trav_to_idx, traj_to_idx, trbv_to_idx, trbd_to_idx, trbj_to_idx, cell_to_idx = to_idx_train

In [66]:
# 重建 params (从映射计算)
params = {
    'n_genes': n_genes,  # 假设 n_genes 全局或从 test_set 计算
    'n_trav': len(trav_to_idx),
    'n_traj': len(traj_to_idx),
    'n_trbv': len(trbv_to_idx),
    'n_trbd': len(trbd_to_idx),
    'n_trbj': len(trbj_to_idx),
    'n_celltype': len(cell_to_idx),
    'n_hpvinf': 2
}

In [68]:
model = TCellClassifier(**params).to(device)
model.load_state_dict(torch.load(f'best_model_fold_{best_fold}.pth'))
model.eval()

TCellClassifier(
  (embed_trav): Embedding(49, 32)
  (embed_traj): Embedding(53, 32)
  (embed_trbv): Embedding(48, 32)
  (embed_trbd): Embedding(4, 32)
  (embed_trbj): Embedding(14, 32)
  (embed_cell): Embedding(17, 8)
  (embed_hpv): Embedding(2, 2)
  (embed_aa_properties): Embedding(6, 32, padding_idx=0)
  (transformer_alpha): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
    )
    (linear1): Linear(in_features=32, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=32, bias=True)
    (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (transformer_beta): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj

In [78]:
# Prediction on unknown
unknown_df = pd.read_csv('./data/unknown_cells_input.csv')
unknown_df['CTgene'] = unknown_df['CTgene'].str.replace('_', '.', regex=False)  # Ensure consistency

In [79]:
# Use training mappings for prediction
to_idx_unknown, X_genes, X_ctgene, X_celltype, X_hpvinf, X_seq_alpha_props, X_seq_beta_props, X_seq_alpha_len, X_seq_beta_len, _ = pretreatment(unknown_df, 
    trav_to_idx=trav_to_idx, traj_to_idx=traj_to_idx, trbv_to_idx=trbv_to_idx, trbd_to_idx=trbd_to_idx, trbj_to_idx=trbj_to_idx, cell_to_idx=cell_to_idx)

In [80]:
unknown_dataset = TCellDataset(X_genes, X_ctgene, X_celltype, X_hpvinf, X_seq_alpha_props, X_seq_beta_props, X_seq_alpha_len, X_seq_beta_len)
unknown_loader = DataLoader(unknown_dataset, batch_size=128, shuffle=False)

In [86]:
model.eval()
all_preds = []
with torch.no_grad():
    for batch in unknown_loader:
        Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl = [x.to(device) for x in batch]
        logits = model(Xg, Xct, Xcell, Xhpv, Xa, Xb, Xal, Xbl)
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape: torch.Size([128, 364])
x_combined shape

In [88]:
unknown_df['predicted_label'] = all_preds
result_df = unknown_df[['ID', 'predicted_label']]
result_df['predicted_label'] = result_df['predicted_label'].map({0: 'nonNeo', 1: 'Neo'})
result_df.to_csv('predicted_unknown_cells_Trans-2.csv', index=False)
print("Prediction completed and results saved to 'predicted_unknown_cells_Trans-2.csv'.")

Prediction completed and results saved to 'predicted_unknown_cells_Trans-2.csv'.


C:\Users\wenzh\AppData\Local\Temp\ipykernel_22360\2318361819.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_df['predicted_label'] = result_df['predicted_label'].map({0: 'nonNeo', 1: 'Neo'})
